# HumanEval Evaluation Example

This notebook demonstrates how to:
1. Extract prompts from HumanEval problems
2. Evaluate code responses
3. Use both standard and lossy compression approaches

In [1]:
! export EVALPLUS_MAX_MEMORY_BYTES=-1


In [2]:
# Setup paths and imports
import sys
import os

# Import the evaluation functions
from lossy_compression.run_human_eval import (
    get_single_problem,
    evaluate_single_response_simple,
    create_claude_model,
    model_answer_single_question
)

/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Extract a Prompt from HumanEval

In [3]:
# Get a specific HumanEval problem
task_id = "HumanEval/1"  # You can change this to any valid task ID

# Extract the problem and expected output
problem, expected_output = get_single_problem(task_id)

# Get the prompt
prompt = problem["prompt"]

print(f"Task ID: {task_id}")
print(f"Entry Point: {problem['entry_point']}")
print("\nPrompt:")
print("="*60)
print(prompt)
print("="*60)

Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl
Task ID: HumanEval/1
Entry Point: separate_paren_groups

Prompt:
from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open brace is properly closed) and not nested within each other
    Ignore any spaces in the input string.
    >>> separate_paren_groups('( ) (( )) (( )( ))')
    ['()', '(())', '(()())']
    """



## 2. Manually Create and Evaluate a Solution

In [4]:
# Example: Manual solution for HumanEval/1
manual_solution = '''
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
'''

# Evaluate the solution
result = evaluate_single_response_simple(task_id, manual_solution)

print(f"✅ Passed: {result['passed']}")
print(f"Status: {result['base_status']}")
if not result['passed']:
    print(f"Failed tests: {result['failed_tests']}")

Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl
✅ Passed: False
Status: timeout
Failed tests: [['( ) (( )) (( )( ))']]


Process Process-1:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/__init__.py", line 143, in unsafe_execute
    reliability_guard(maximum_memory_bytes=query_maximum_memory_bytes())
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/utils.py", line 118, in reliability_guard
    resource.setrlimit(
    ~~~~~~~~~~~~~~~~~~^
        res

In [5]:
raise 

RuntimeError: No active exception to reraise

## 3. Generate Solution Using Claude

In [ ]:
# Create a Claude model
model = create_claude_model(
    model_name="claude-3-haiku-20240307",  # Can use haiku, sonnet, or opus
    temperature=0.0
)

# Generate a solution
responses = model_answer_single_question(
    model=model,
    prompt=prompt,
    greedy=True,
    n_samples=1
)

generated_solution = responses[0]
print("Generated Solution:")
print("="*60)
print(generated_solution)
print("="*60)

Creating Claude model: claude-3-haiku-20240307
Generating 1 sample(s) for prompt: from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input t...
Generated 1 response(s)
Generated Solution:
def separate_paren_groups(paren_string: str) -> List[str]:
    result = []
    stack = []
    current_group = ""

    for char in paren_string:
        if char == "(":
            stack.append(char)
            current_group += char
        elif char == ")":
            stack.pop()
            current_group += char
            if not stack:
                result.append(current_group)
                current_group = ""
    return result


In [ ]:
# Evaluate the generated solution
result = evaluate_single_response_simple(task_id, generated_solution)

print(f"\n📊 Evaluation Results:")
print(f"{'='*40}")
if result['passed']:
    print(f"✅ SUCCESS: All test cases passed!")
else:
    print(f"❌ FAILURE: Some test cases failed")
    if result['failed_tests']:
        print(f"Failed on inputs: {result['failed_tests'][:3]}...")  # Show first 3 failed tests

print(f"\nStatus: {result['base_status']}")

Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl

📊 Evaluation Results:
❌ FAILURE: Some test cases failed
Failed on inputs: [['( ) (( )) (( )( ))']]...

Status: timeout


Process Process-2:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/__init__.py", line 143, in unsafe_execute
    reliability_guard(maximum_memory_bytes=query_maximum_memory_bytes())
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/utils.py", line 118, in reliability_guard
    resource.setrlimit(
    ~~~~~~~~~~~~~~~~~~^
        res

## 4. Test Multiple Problems

In [ ]:
# Test multiple HumanEval problems
test_tasks = [1, 2, 5, 10, 15]  # HumanEval task numbers
results = []

for task_num in test_tasks:
    task_id = f"HumanEval/{task_num}"
    
    # Get problem
    problem, _ = get_single_problem(task_id)
    prompt = problem["prompt"]
    
    # Generate solution
    responses = model_answer_single_question(
        model=model,
        prompt=prompt,
        greedy=True,
        n_samples=1
    )
    
    # Evaluate
    result = evaluate_single_response_simple(task_id, responses[0])
    results.append(result)
    
    # Print result
    status = "✅" if result['passed'] else "❌"
    print(f"{status} {task_id}: {result['base_status']}")

# Summary
passed = sum(1 for r in results if r['passed'])
total = len(results)
print(f"\n📊 Summary: {passed}/{total} passed ({passed/total*100:.1f}%)")

Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl
Generating 1 sample(s) for prompt: from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input t...
Generated 1 response(s)
Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl
❌ HumanEval/1: timeout
Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_2.pkl
Generating 1 sample(s) for prompt: 

def truncate_number(number: float) -> float:
    """ Given a positive floating point number, it ca...
Generated 1 response(s)
Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_2.pkl
❌ HumanEval/2: timeout
Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_5.pkl
Generating 1 sample(s) for prompt: from typing import List


def int

## 5. Using Lossy Compression (Q&A Method)

In [ ]:
# Import lossy compression functions
from lossy_compression.SLM_question_answering_compression import batch_questions_SLM_generation

# Get a problem to solve
task_id = "HumanEval/1"
problem, _ = get_single_problem(task_id)
prompt = problem["prompt"]

# Add instruction for code generation
code_prompt = f"""Complete this Python function. Return ONLY the complete function implementation including the signature, no explanations or markdown:

{prompt}"""

print("Running Q&A compression method...")
print("This will:")
print("1. Generate initial solution with SLM")
print("2. Generate clarifying questions")
print("3. Answer questions to improve understanding")
print("4. Generate final improved solution")
print("="*60)

In [ ]:
# Run the Q&A compression
final_code, qa_pairs, metrics = batch_questions_SLM_generation(
    code_prompt,
    large_model_name="claude-3-opus-20240229",  # LLM for guidance
    small_model_name="claude-3-haiku-20240307",  # SLM for generation
    question_model_name="claude-3-5-sonnet-20241022",  # Model for questions
    num_questions=10,  # Number of Q&A pairs
    verbose=True
)

print(f"\n📊 Quality Scores:")
print(f"Initial: {metrics['initial_score']}/10")
print(f"After Q&A: {metrics['qa_only_score']}/10")
print(f"Final: {metrics['final_score']}/10")
print(f"Improvement: +{metrics['improvement_total']}")

🚀 Starting BATCH question generation...
📝 Generating 10 questions upfront
Starting iterative SLM loop...
Small model: claude-3-haiku-20240307
Question generation model: claude-3-5-sonnet-20241022
🌐 Using remote API model: claude-3-haiku-20240307

📝 Step 1: Generating initial answers...
✅ Initial LLM answer: from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    result = []
    current_group = []
    balance = 0
    
    for char in paren_string:
        if char == '(':
            balance += 1
            current_group.append(char)
        elif char == ')':
            balance -= 1
            current_group.append(char)
            
            if balance == 0:
                result.append(''.join(current_group))
                current_group = []
    
    return result
--------------------------------
✅ Initial SLM answer: Here's the completed Python function:

```python
from typing import List

def separate_paren_groups(paren_string: str) -> List[st

In [ ]:
# Evaluate the compressed solution
result = evaluate_single_response_simple(task_id, final_code)

print(f"\n📊 HumanEval Evaluation:")
print(f"{'='*40}")
if result['passed']:
    print(f"✅ SUCCESS: Solution passed all tests!")
else:
    print(f"❌ FAILURE: Solution failed some tests")
    if result['failed_tests']:
        print(f"Failed on: {result['failed_tests'][:3]}...")

# Show the final solution
print(f"\nFinal Solution:")
print("="*60)
print(final_code)
print("="*60)

Load from ground-truth from /Users/roy/Library/Caches/evalplus/fe585eb4df8c88d844eeb463ea4d0302_HumanEval_1.pkl

📊 HumanEval Evaluation:
❌ FAILURE: Solution failed some tests
Failed on: [['( ) (( )) (( )( ))']]...

Final Solution:
Here is the completed Python function that incorporates the additional information provided:

```python
from typing import List

def separate_paren_groups(paren_string: str) -> List[str]:
    result = []
    stack = []
    current_group = ""

    for char in paren_string:
        if char == '(':
            stack.append(char)
            current_group += char
        elif char == ')':
            if stack:
                stack.pop()
                current_group += char
                if not stack:
                    result.append(current_group)
                    current_group = ""
        elif char != ' ':
            current_group += char

    return result
```


Process Process-8:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/__init__.py", line 143, in unsafe_execute
    reliability_guard(maximum_memory_bytes=query_maximum_memory_bytes())
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/utils.py", line 118, in reliability_guard
    resource.setrlimit(
    ~~~~~~~~~~~~~~~~~~^
        res

## 6. Compare Different Approaches

## 7. Utility Functions

In [ ]:
def quick_test(task_num: int, solution_code: str):
    """Quick function to test any solution against a HumanEval problem."""
    task_id = f"HumanEval/{task_num}"
    result = evaluate_single_response_simple(task_id, solution_code)
    
    if result['passed']:
        print(f"✅ Task {task_num}: PASSED")
    else:
        print(f"❌ Task {task_num}: FAILED")
        if result['failed_tests']:
            print(f"   Failed on: {result['failed_tests'][:2]}")
    
    return result['passed']

# Example usage
test_solution = '''
def add(a, b):
    return a + b
'''

# Test against HumanEval/53 (which is an add function)
quick_test(53, test_solution)

Computing expected output...
Expected outputs computed in 0.00s
❌ Task 53: FAILED
   Failed on: [[371, 167]]


Process Process-9:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/__init__.py", line 143, in unsafe_execute
    reliability_guard(maximum_memory_bytes=query_maximum_memory_bytes())
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/roy/code/research/model-compression-bit-limiting/.venv/lib/python3.13/site-packages/evalplus/eval/utils.py", line 118, in reliability_guard
    resource.setrlimit(
    ~~~~~~~~~~~~~~~~~~^
        res

False